# Mux Readout Workflow

Generate a slot-stable mux config and run mux time-of-flight with the PFB readout channels.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from qick.pyro import make_proxy
import Pyro4

from QickworkspaceV2.config.system_cfg import DATA_PATH, config_list
from QickworkspaceV2.tools.system_tool import ExperimentConfig
from QickworkspaceV2.core.base_experiment import BaseExperiment
from QickworkspaceV2.experiments_mux import MuxTOF

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

In [ ]:
ns_host = "192.168.10.82"
ns_port = 8888
proxy_name = "myqick"

soc, soccfg = make_proxy(ns_host=ns_host, ns_port=ns_port, proxy_name=proxy_name)
BaseExperiment.setup(soc, soccfg, DATA_PATH)
print(soccfg)

In [ ]:
config_all = ExperimentConfig(config_list)
config_all.qubit_names()

In [ ]:
arm_qubit = ["Q1", "Q3", "Q4"]

run_cfg = config_all.muxconfig(
    arm_qubit,
    mux_gen=4,
    mux_ro_ch_start=2,
    LO_ext=6000,
)
run_cfg["mux_nqz"] = 1

summary_keys = [
    "all_qubit_names",
    "qubit_names",
    "active_slots",
    "mask",
    "res_ch",
    "ro_chs",
    "active_ro_chs",
    "res_freqs",
    "res_gains",
    "res_phases",
    "ro_phases",
    "mixer_freq",
    "LO_ext",
    "trig_time",
    "mux_scale_count",
]

for key in summary_keys:
    print(f"{key}: {run_cfg.get(key)}")

The mux mask is fixed to the number of configured qubit slots. Non-armed qubits keep their frequency slot but get zero readout gain.

In [ ]:
tof = MuxTOF(run_cfg)
tof.prog_asm()

In [ ]:
result = tof.run(py_avg=100)
result.fit_result

In [ ]:
# Optional: inspect the averaged complex traces manually.
if tof.iqdata is not None:
    fig, axes = plt.subplots(len(run_cfg["qubit_names"]), 1, figsize=(8, 3 * len(run_cfg["qubit_names"])), squeeze=False)
    for ax, name, trace in zip(axes[:, 0], run_cfg["qubit_names"], tof.iqdata):
        ax.plot(tof.t, trace.real, label="I")
        ax.plot(tof.t, trace.imag, label="Q")
        ax.plot(tof.t, np.abs(trace), label="abs")
        ax.set_title(name)
        ax.set_xlabel("us")
        ax.legend()
    plt.tight_layout()